In [1]:
import os
import json
import time
import argparse
import warnings
warnings.filterwarnings("ignore")

import numpy as np

try:
    import cv2
except ImportError:
    raise SystemExit(
        "opencv-python is required for this script. Install with:\n"
        "  pip install opencv-python"
    )

try:
    import mediapipe as mp
except ImportError:
    raise SystemExit(
        "mediapipe is required for this script. Install with:\n"
        "  pip install mediapipe"
    )

import tensorflow as tf
from tensorflow.keras.models import load_model

SEQUENCE_LENGTH = 30
COUNTDOWN_SECONDS = 5.0  # matches test_webcam_trigger.py exactly

mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils

In [2]:
# ============================================================
# Exact extraction logic from extraction.py / test_webcam_trigger.py
# ============================================================
SELECTED_FACE_IDS = [
    # Lips (For mouthing/shape)
    0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95, 146,
    178, 181, 191, 267, 269, 270, 291, 308, 310, 311, 312, 314, 317, 318,
    321, 324, 375, 402, 405, 415,
    # Eyebrows
    46, 52, 53, 55, 65, 70, 105, 107, 276, 282, 283, 285, 295, 300, 334, 336,
    # Left Cheek Zone
    50, 118, 123, 137, 205, 206, 207, 212, 214, 216,
    # Right Cheek Zone
    280, 347, 352, 366, 425, 426, 427, 432, 434, 436
]


def extract_and_normalize_keypoints(results):
    """Verbatim from test_webcam_trigger.py / extraction.py."""
    cx, cy, cz = 0.0, 0.0, 0.0
    scale = 1.0

    if results.pose_landmarks:
        l_sh = results.pose_landmarks.landmark[11]
        r_sh = results.pose_landmarks.landmark[12]
        cx, cy, cz = (l_sh.x + r_sh.x) / 2, (l_sh.y + r_sh.y) / 2, (l_sh.z + r_sh.z) / 2
        shoulder_dist = np.linalg.norm([l_sh.x - r_sh.x, l_sh.y - r_sh.y, l_sh.z - r_sh.z])
        if shoulder_dist > 0:
            scale = shoulder_dist

    def norm(lm_list, is_face=False):
        if not lm_list:
            return np.zeros(len(SELECTED_FACE_IDS) * 3) if is_face else np.zeros(21 * 3)
        data = []
        for i, lm in enumerate(lm_list.landmark):
            if is_face and i not in SELECTED_FACE_IDS:
                continue
            data.extend([(lm.x - cx) / scale, (lm.y - cy) / scale, (lm.z - cz) / scale])
        return np.array(data)

    lh, rh = norm(results.left_hand_landmarks), norm(results.right_hand_landmarks)
    face = norm(results.face_landmarks, is_face=True)

    if results.pose_landmarks:
        pose = np.array([[(lm.x - cx) / scale, (lm.y - cy) / scale, (lm.z - cz) / scale]
                          for lm in results.pose_landmarks.landmark]).flatten()
    else:
        pose = np.zeros(33 * 3)

    return np.concatenate([pose, face, lh, rh])

In [3]:
# ============================================================
# Live capture loop (mirrors test_webcam_trigger.py's state machine)
# ============================================================
def capture_one_sequence(holistic):
    """Runs the SPACE -> countdown -> RECORDING flow exactly like
    test_webcam_trigger.py. Returns (sequence, processing_ms, wallclock_ms)
    or None if the user quit."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam (source 0).")

    state = "IDLE"
    countdown_start_time = 0
    sequence = []
    processing_time_ms = 0.0   # MediaPipe + normalization compute only
    wallclock_start = None

    print("Press SPACE to start recording, 'q' to quit.")

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("Failed to read from webcam.")
                return None

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            if state == "RECORDING":
                t0 = time.perf_counter()
                results = holistic.process(image)
                keypoints = extract_and_normalize_keypoints(results)
                t1 = time.perf_counter()
                processing_time_ms += (t1 - t0) * 1000
                sequence.append(keypoints)
            else:
                results = holistic.process(image)  # still needed to draw / for UX

            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

            if state == "RECORDING":
                cv2.rectangle(image, (0, 0), (640, 40), (0, 0, 255), -1)
                cv2.putText(image, f'RECORDING: {len(sequence)}/{SEQUENCE_LENGTH} frames',
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
                if len(sequence) == SEQUENCE_LENGTH:
                    wallclock_ms = (time.perf_counter() - wallclock_start) * 1000
                    cap.release()
                    cv2.destroyAllWindows()
                    return np.array(sequence, dtype=np.float32), processing_time_ms, wallclock_ms

            elif state == "IDLE":
                cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
                cv2.putText(image, "Press SPACE to start recording", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            elif state == "COUNTDOWN":
                time_left = COUNTDOWN_SECONDS - (time.time() - countdown_start_time)
                if time_left <= 0:
                    state = "RECORDING"
                    sequence = []
                    processing_time_ms = 0.0
                    wallclock_start = time.perf_counter()
                else:
                    cv2.rectangle(image, (0, 0), (640, 80), (0, 165, 255), -1)
                    cv2.putText(image, f"GET READY: {int(time_left) + 1}", (10, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3, cv2.LINE_AA)

            cv2.imshow('SignLingo Latency Benchmark', image)
            key = cv2.waitKey(10) & 0xFF
            if key == ord('q'):
                cap.release()
                cv2.destroyAllWindows()
                return None
            elif key == ord(' ') and state == "IDLE":
                state = "COUNTDOWN"
                countdown_start_time = time.time()
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [4]:
# ============================================================
# Model timing (same captured sequence, every format)
# ============================================================
def time_keras_predict(h5_path, sequence, n_repeats=5):
    model = load_model(h5_path)
    inp = np.expand_dims(sequence, axis=0)
    latencies = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        model.predict(inp, verbose=0)
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
    return float(np.mean(latencies))


def time_tflite_predict(tflite_path, sequence, n_repeats=5):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    inp = sequence[np.newaxis, ...].astype(np.float32)

    latencies = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details[0]["index"], inp)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details[0]["index"])
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
    return float(np.mean(latencies))

In [6]:
# ============================================================
# Main
# ============================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_sequences", type=int, default=10,
                        help="Number of live sequences to capture and average over "
                             "(kept >1 so latency is reported as mean +/- std, not a "
                             "single sample)")
    parser.add_argument("--h5_model", default="models/signlingo_gru_best.h5",
                        help="Path to the raw Keras .h5 model")
    args, _ = parser.parse_known_args()  # ignore Jupyter kernel's --f=... arg

    candidates_tflite = [
        ("Float32 (baseline)", "signlingo_fp32.tflite"),
        ("Float16",            "signlingo_fp16.tflite"),
        ("Dynamic Range",      "signlingo_dynamic.tflite"),
        ("Full INT8",          "signlingo_int8.tflite"),
    ]

    processing_times, wallclock_times, sequences = [], [], []

    with mp_holistic.Holistic(min_detection_confidence=0.5,
                               min_tracking_confidence=0.5) as holistic:
        for i in range(args.n_sequences):
            print(f"\n--- Capturing sequence {i + 1}/{args.n_sequences} ---")
            result = capture_one_sequence(holistic)
            if result is None:
                print("Quit before completing capture.")
                return
            seq, proc_ms, wall_ms = result
            sequences.append(seq)
            processing_times.append(proc_ms)
            wallclock_times.append(wall_ms)
            print(f"  Extraction processing time: {proc_ms:.2f} ms  |  "
                  f"Capture wall-clock: {wall_ms:.2f} ms")

    mean_processing = float(np.mean(processing_times))
    std_processing  = float(np.std(processing_times))
    mean_wallclock  = float(np.mean(wallclock_times))

    print(f"\nProcessing time (compute only), N={len(processing_times)}: "
          f"{mean_processing:.2f} +/- {std_processing:.2f} ms")
    print(f"Mean capture wall-clock time (incl. camera pacing): {mean_wallclock:.2f} ms\n")

    results = []

    def add_row(label, model_times):
        # Paired per-sequence totals (same N captured sequences), not a
        # combination of separately-averaged means -- gives a statistically
        # correct mean +/- std on the actual end-to-end number.
        totals = [p + m for p, m in zip(processing_times, model_times)]
        model_mean, model_std = float(np.mean(model_times)), float(np.std(model_times))
        total_mean, total_std = float(np.mean(totals)), float(np.std(totals))
        results.append({
            "Model": label,
            "N": len(model_times),
            "Processing Mean (ms)": round(mean_processing, 3),
            "Processing Std (ms)": round(std_processing, 3),
            "Model Inference Mean (ms)": round(model_mean, 3),
            "Model Inference Std (ms)": round(model_std, 3),
            "Total Processing Mean (ms)": round(total_mean, 3),
            "Total Processing Std (ms)": round(total_std, 3),
            "Total incl. Capture Wall-Clock (ms)": round(mean_wallclock + model_mean, 3),
        })
        print(f"  Model inference: {model_mean:.3f} +/- {model_std:.3f} ms  |  "
              f"Total: {total_mean:.3f} +/- {total_std:.3f} ms")

    if os.path.exists(args.h5_model):
        print(f"Timing raw Keras model: {args.h5_model} …")
        add_row("Raw Keras .h5 (eager)", [time_keras_predict(args.h5_model, seq) for seq in sequences])
    else:
        print(f"  Skipping raw Keras model: '{args.h5_model}' not found")

    for label, path in candidates_tflite:
        if not os.path.exists(path):
            print(f"  Skipping {label}: '{path}' not found")
            continue
        print(f"Timing model inference: {label} …")
        add_row(label, [time_tflite_predict(path, seq) for seq in sequences])

    print("\n── End-to-End Latency Results (mean +/- std) ─────────────────")
    header = (f"{'Model':<24}{'Processing (ms)':>20}{'Model (ms)':>18}{'Total (ms)':>20}")
    print(header)
    for r in results:
        proc = f"{r['Processing Mean (ms)']:.3f}+/-{r['Processing Std (ms)']:.3f}"
        mdl  = f"{r['Model Inference Mean (ms)']:.3f}+/-{r['Model Inference Std (ms)']:.3f}"
        tot  = f"{r['Total Processing Mean (ms)']:.3f}+/-{r['Total Processing Std (ms)']:.3f}"
        print(f"{r['Model']:<24}{proc:>20}{mdl:>18}{tot:>20}")

    with open("e2e_latency_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("\nSaved → e2e_latency_results.json")


if __name__ == "__main__":
    main()


--- Capturing sequence 1/10 ---
Press SPACE to start recording, 'q' to quit.
  Extraction processing time: 1694.92 ms  |  Capture wall-clock: 2357.15 ms

--- Capturing sequence 2/10 ---
Press SPACE to start recording, 'q' to quit.
  Extraction processing time: 1613.37 ms  |  Capture wall-clock: 2289.44 ms

--- Capturing sequence 3/10 ---
Press SPACE to start recording, 'q' to quit.
  Extraction processing time: 1603.56 ms  |  Capture wall-clock: 2282.47 ms

--- Capturing sequence 4/10 ---
Press SPACE to start recording, 'q' to quit.
  Extraction processing time: 1489.17 ms  |  Capture wall-clock: 2193.44 ms

--- Capturing sequence 5/10 ---
Press SPACE to start recording, 'q' to quit.
  Extraction processing time: 1485.80 ms  |  Capture wall-clock: 2056.01 ms

--- Capturing sequence 6/10 ---
Press SPACE to start recording, 'q' to quit.
  Extraction processing time: 1489.44 ms  |  Capture wall-clock: 2093.62 ms

--- Capturing sequence 7/10 ---
Press SPACE to start recording, 'q' to quit

  Model inference: 107.309 +/- 5.665 ms  |  Total: 1634.189 +/- 80.057 ms
Timing model inference: Float32 (baseline) …
  Model inference: 1.458 +/- 0.240 ms  |  Total: 1528.338 +/- 76.505 ms
Timing model inference: Float16 …
  Model inference: 1.574 +/- 0.069 ms  |  Total: 1528.454 +/- 76.318 ms
Timing model inference: Dynamic Range …
  Model inference: 1.083 +/- 0.051 ms  |  Total: 1527.963 +/- 76.367 ms
Timing model inference: Full INT8 …
  Model inference: 1.224 +/- 0.032 ms  |  Total: 1528.104 +/- 76.338 ms

── End-to-End Latency Results (mean +/- std) ─────────────────
Model                        Processing (ms)        Model (ms)          Total (ms)
Raw Keras .h5 (eager)      1526.880+/-76.337   107.309+/-5.665   1634.189+/-80.057
Float32 (baseline)         1526.880+/-76.337     1.458+/-0.240   1528.338+/-76.505
Float16                    1526.880+/-76.337     1.574+/-0.069   1528.454+/-76.318
Dynamic Range              1526.880+/-76.337     1.083+/-0.051   1527.963+/-76.367
Full